In [ ]:
import pandas as pd

# Step 1: Load the dataset
df = pd.read_csv("s01.csv")  # Loading the EEG data

# Step 2: Display basic dataset info
print("Dataset Information:\n")
print(df.info())  # Shows number of rows, columns, and data types

# Step 3: Show first 5 rows of data
print("\nFirst few rows of the dataset:\n")
print(df.head())  # Helps us see how the data looks

Dataset Information:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30999 entries, 0 to 30998
Data columns (total 19 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   13.99   30999 non-null  float64
 1   10.55   30999 non-null  float64
 2   14.146  30999 non-null  float64
 3   13.902  30999 non-null  float64
 4   14.289  30999 non-null  float64
 5   14.455  30999 non-null  float64
 6   10.398  30999 non-null  float64
 7   7.6692  30999 non-null  float64
 8   15.74   30999 non-null  float64
 9   15.315  30999 non-null  float64
 10  10.047  30999 non-null  float64
 11  6.5063  30999 non-null  float64
 12  9.8192  30999 non-null  float64
 13  13.829  30999 non-null  float64
 14  8.9212  30999 non-null  float64
 15  17.561  30999 non-null  float64
 16  16.089  30999 non-null  float64
 17  18.206  30999 non-null  float64
 18  11.392  30999 non-null  float64
dtypes: float64(19)
memory usage: 4.5 MB
None

First few rows of the dataset:

    13.99    

In [ ]:
# Step 4: Check for missing values
print("\nMissing values in the dataset:\n")
print(df.isnull().sum())  # This will show if any column has empty (NaN) values

# Step 5: Get summary statistics of EEG signals
print("\nStatistical summary:\n")
print(df.describe())  # Gives information like mean, min, max, etc.



Missing values in the dataset:

13.99     0
10.55     0
14.146    0
13.902    0
14.289    0
14.455    0
10.398    0
7.6692    0
15.74     0
15.315    0
10.047    0
6.5063    0
9.8192    0
13.829    0
8.9212    0
17.561    0
16.089    0
18.206    0
11.392    0
dtype: int64

Statistical summary:

              13.99         10.55        14.146        13.902        14.289  \
count  30999.000000  30999.000000  30999.000000  30999.000000  30999.000000   
mean       0.126307      0.116256     -0.105460      0.155934     -0.058885   
std       13.933224     10.565007     12.020449     11.119948     15.295288   
min      -53.465000    -43.343000    -63.668000    -45.775000    -60.671000   
25%       -7.951650     -6.276050     -7.168100     -6.961350     -8.805750   
50%        0.000859      0.203280     -0.051719      0.163230     -0.000461   
75%        8.074500      6.819150      7.294600      7.728150      9.022800   
max       69.610000     58.349000     52.693000     65.220000     54.54

In [ ]:
import numpy as np
import scipy.signal as signal

# Step 6: Function to compute power in different frequency bands
def compute_bandpower(data, sf=250, band=(4, 8)):
    """Calculate the power of a specific EEG frequency band using Welch's method."""
    freqs, psd = signal.welch(data, sf, nperseg=1024)  # Compute Power Spectral Density
    idx_band = np.logical_and(freqs >= band[0], freqs <= band[1])  # Select frequency range
    return np.trapz(psd[idx_band], freqs[idx_band])  # Compute total power in this band

# Step 7: Extract EEG frequency features (Theta, Beta, Gamma)
features = []
for col in df.columns:  # Go through each EEG channel
    theta_power = compute_bandpower(df[col], band=(4, 8))  # Theta Band (Memory)
    beta_power = compute_bandpower(df[col], band=(14, 30))  # Beta Band (Cognitive load)
    gamma_power = compute_bandpower(df[col], band=(30, 100))  # Gamma Band (Memory recall)

    theta_beta_ratio = theta_power / beta_power if beta_power != 0 else 0  # Avoid division by zero
    theta_gamma_ratio = theta_power / gamma_power if gamma_power != 0 else 0

    features.append([theta_power, beta_power, gamma_power, theta_beta_ratio, theta_gamma_ratio])

# Step 8: Convert extracted features into a DataFrame
feature_df = pd.DataFrame(features, columns=['Theta Power', 'Beta Power', 'Gamma Power', 'Theta/Beta', 'Theta/Gamma'])


In [ ]:
# Step 9: Generate random labels for now (Replace with real task performance data if available)
np.random.seed(42)  # Set random seed for consistency
feature_df['Label'] = np.random.choice([0, 1], size=len(feature_df))  # 1 = Memory Intact, 0 = Non-Intact


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Step 10: Split the data into training and testing sets
X = feature_df.drop(columns=['Label'])  # Features (Theta, Beta, Gamma)
y = feature_df['Label']  # Target labels (Memory performance)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 11: Scale the data (Normalize EEG features for better learning)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 12: Train a Random Forest model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)  # Train the model on EEG features

# Step 13: Make predictions
y_pred = clf.predict(X_test)

# Step 14: Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")  # Show accuracy
print(classification_report(y_test, y_pred))  # Detailed report


Model Accuracy: 50.00%
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       0.00      0.00      0.00         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import pandas as pd
import numpy as np
import scipy.signal as signal

# Load EEG dataset
df = pd.read_csv("s01.csv")

# Check for missing values
df = df.dropna()  # Drop rows with missing values


In [ ]:
# Function to compute power in EEG frequency bands
def compute_bandpower(data, sf=250, band=(4, 8)):
    """Calculate power in a specific frequency band using Welch's method"""
    freqs, psd = signal.welch(data, sf, nperseg=1024)
    idx_band = np.logical_and(freqs >= band[0], freqs <= band[1])
    return np.trapz(psd[idx_band], freqs[idx_band])  # Compute total power in band

# Extract EEG frequency features for all channels
features = []
for col in df.columns:  # Iterate through EEG channels
    theta = compute_bandpower(df[col], band=(4, 8))
    alpha = compute_bandpower(df[col], band=(8, 12))
    beta = compute_bandpower(df[col], band=(14, 30))
    gamma = compute_bandpower(df[col], band=(30, 100))

    # Compute ratios
    theta_beta_ratio = theta / beta if beta != 0 else 0
    theta_gamma_ratio = theta / gamma if gamma != 0 else 0
    alpha_beta_ratio = alpha / beta if beta != 0 else 0

    features.append([theta, alpha, beta, gamma, theta_beta_ratio, theta_gamma_ratio, alpha_beta_ratio])

# Convert features into a DataFrame
feature_df = pd.DataFrame(features, columns=['Theta', 'Alpha', 'Beta', 'Gamma', 'Theta/Beta', 'Theta/Gamma', 'Alpha/Beta'])


In [ ]:
# Generate realistic labels (simulate memory performance)
feature_df['Label'] = feature_df['Theta/Beta'].apply(lambda x: 1 if x > 0.5 else 0)  # 1 = Intact, 0 = Non-Intact


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Prepare Data for LSTM
X = feature_df.drop(columns=['Label']).values  # Features
y = feature_df['Label'].values  # Labels

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for LSTM (samples, timesteps, features)
X = X.reshape((X.shape[0], 1, X.shape[1]))

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define LSTM Model
model = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(1, X.shape[2])),
    LSTM(32, activation='relu', return_sequences=False),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=4, validation_data=(X_test, y_test), verbose=1)

# Evaluate model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 172ms/step - accuracy: 0.5767 - loss: 0.6933 - val_accuracy: 1.0000 - val_loss: 0.6883
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.6835 - val_accuracy: 1.0000 - val_loss: 0.6790
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.6747 - val_accuracy: 1.0000 - val_loss: 0.6698
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 0.6649 - val_accuracy: 1.0000 - val_loss: 0.6602
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.6540 - val_accuracy: 1.0000 - val_loss: 0.6498
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.6442 - val_accuracy: 1.0000 - val_loss: 0.6387
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6313 - val_accuracy: 1.0000 - val_loss: 0.6269
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 1.0000 - loss: 0.6177 - val_accuracy: 1.0000 - val_loss: 0.6139
Epoch 9/50

In [ ]:
from sklearn.metrics import classification_report

# Make predictions
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)  # Convert probabilities to 0 or 1

# Print classification report
print(classification_report(y_test, y_pred))


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 567ms/step
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         4

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



In [ ]:
print(np.unique(y_test, return_counts=True))  # Check count of 0s and 1s


(array([1]), array([4]))


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
import numpy as np
print(np.unique(y_test, return_counts=True))  # Count of labels in test set


(array([1]), array([4]))


In [ ]:
print("Overlap between train & test:", np.any(np.isin(X_train.flatten(), X_test.flatten())))


Overlap between train & test: False


In [ ]:
import numpy as np

# Check class distribution in y_test
unique_labels, label_counts = np.unique(y_test, return_counts=True)
print("Labels:", unique_labels)
print("Counts:", label_counts)


Labels: [1]
Counts: [4]


In [ ]:
from sklearn.model_selection import train_test_split

# Ensure balanced classes in train & test
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)  # Ensures equal distribution


In [ ]:
import numpy as np

# Example EEG sample (random values, replace with real EEG data)
sample_input = np.random.rand(1, X_train.shape[1], X_train.shape[2])  # Reshape to match LSTM input shape

# Make prediction
prediction = model.predict(sample_input)
predicted_class = (prediction > 0.5).astype(int)  # Convert probability to class (0 or 1)

print("Predicted Class (Memory Intact = 1, Non-Intact = 0):", predicted_class[0][0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
Predicted Class (Memory Intact = 1, Non-Intact = 0): 1
